# Judge / Extractor Agreement

Loads the artifacts written by `examples/scripts/evaluation/*.py` into `results/agreement_analysis/` and re-plots them with a consistent, publication-friendly style. See `examples/scripts/evaluation/README.md` for what each metric means.

Run the evaluation scripts first if `results/agreement_analysis/` is empty or stale.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from llm_synthesis.utils.style_utils import get_cmap, get_palette, set_style

cmap = get_cmap()
palette = get_palette()
set_style()

# Continuous cmap for heatmaps, interpolated from the brand palette
# (get_cmap() is a discrete ListedColormap, fine for bars but blocky on heatmaps).
heat_cmap = LinearSegmentedColormap.from_list(
    "brand_sequential", [palette[6], palette[0], palette[2]]
)

RESULTS_DIR = Path("../../results/agreement_analysis")
assert RESULTS_DIR.exists(), (
    f"Run the eval scripts first: {RESULTS_DIR} not found"
)

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)


def load_csv(name):
    return pd.read_csv(RESULTS_DIR / name)


def load_json(name):
    with open(RESULTS_DIR / name) as fh:
        return json.load(fh)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")

## Extractor x Judge score matrix

Mean `overall_score` each judge LLM assigns to each extractor LLM (`insights_judge_extractor_matrix.csv`). Diagonal = self-scoring.

In [ ]:
matrix = load_csv("insights_judge_extractor_matrix.csv").set_index("synth_llm")
matrix.index.name = "extractor"
matrix.columns.name = "judge"

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
ax.set_title("Extractor x Judge overall_score (diagonal = self-scoring)")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge")
plt.show()

## Inter-judge agreement (Spearman)

`insights_interjudge_spearman.csv` — how similarly the judge LLMs rank the same extractions relative to each other.

In [ ]:
import numpy as np

spearman = load_csv("insights_interjudge_spearman.csv").set_index("Unnamed: 0")
spearman.index.name = None
mask = np.triu(np.ones_like(spearman, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    spearman,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    mask=mask,
    cbar_kws={"label": "Spearman rho"},
    ax=ax,
)
ax.set_title("Inter-judge rank correlation")
plt.tight_layout()
savefig(fig, "heatmap_interjudge_spearman")
plt.show()

## Agreement with human ground truth, by ranking metric

`multi_llm_judge_ranking_{abs_diff,rho,kappa,icc2,icc3}.json`, one file per `--rank-by` run of `compare_multi_llm_results_complete.py`.

In [ ]:
METRICS = ["abs_diff", "rho", "kappa", "icc2", "icc3"]

ranking = pd.concat(
    [
        pd.DataFrame(load_json(f"multi_llm_judge_ranking_{m}.json"))
        for m in METRICS
        if (RESULTS_DIR / f"multi_llm_judge_ranking_{m}.json").exists()
    ]
)
rank_pivot = ranking.pivot_table(
    index="judge", columns="rank_by", values="rank"
)
diff_pivot = ranking.set_index(["judge", "rank_by"])["abs_diff"].unstack()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(
    rank_pivot,
    annot=True,
    fmt=".0f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "rank (1 = best)"},
    ax=axes[0],
)
axes[0].set_title("Judge rank by metric")

sns.heatmap(
    diff_pivot,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "abs_diff vs. human"},
    ax=axes[1],
)
axes[1].set_title("|judge - human| overall_score, per rank-by run")
plt.tight_layout()
savefig(fig, "heatmap_judge_ranking_by_metric")
plt.show()

**Which metric to trust:** `abs_diff` (raw score gap) is confounded by a judge's overall leniency/harshness — a judge can be perfectly rank-consistent with human and still score poorly on `abs_diff` if it just runs on a shifted scale (see `mean_diff` above; judges range from -0.6 to +0.5). `kappa` is noisy at this sample size. `rho` (Spearman) is scale-invariant and checks ranking only. **ICC2 is the most defensible single metric for "agreement with human"**: it's explicitly designed for raters who may differ in scale/offset, which is exactly the situation here. We rank/justify judge choice primarily on ICC2 (`loo_no_self`), with rho as a secondary rank-based check.

## Self-preference bias

Does a judge score its own extractions higher than it scores others'? `insights_self_preference.csv` (self vs. peer mean) and `insights_self_bias_did.csv` (difference-in-differences vs. human).

In [ ]:
self_pref = load_csv("insights_self_preference.csv")
self_bias = load_csv("insights_self_bias_did.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_df = self_pref.melt(
    id_vars="model",
    value_vars=["self_score_mean", "peer_score_mean"],
    var_name="target",
    value_name="score",
)
sns.barplot(
    data=plot_df,
    x="model",
    y="score",
    hue="target",
    ax=axes[0],
    palette=palette[:2],
)
axes[0].set_title("Self vs. peer scoring")
axes[0].set_ylabel("mean overall_score")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=self_bias, x="model", y="self_bias_did", ax=axes[1], color=palette[0]
)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Self-bias (DiD vs. human)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig(fig, "bar_self_preference_bias")
plt.show()

## Score dimension breakdown: judges vs. human

`insights_dimension_means.csv` — mean score per rubric dimension, judges pooled vs. human. `insights_judge_behavior.csv` breaks the same dimensions out per judge (not pooled), which lets us also show `abs_diff` per judge per dimension vs. the `HUMAN` row.

In [ ]:
dims = load_csv("insights_dimension_means.csv").melt(
    id_vars="dimension", var_name="source", value_name="mean_score"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=dims,
    y="dimension",
    x="mean_score",
    hue="source",
    ax=ax,
    palette=palette[:2],
)
ax.set_xlim(0, 5)
ax.set_title("Judges (pooled) vs. human, by rubric dimension")
plt.tight_layout()
savefig(fig, "bar_dimension_means")
plt.show()

In [ ]:
dimension_score_cols = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

behavior_dims = load_csv("insights_judge_behavior.csv").set_index("judge")[
    dimension_score_cols
]
behavior_dims.columns = [
    c.replace("_score", "").replace("_", " ") for c in behavior_dims.columns
]

human_row = behavior_dims.loc["HUMAN"]
judge_dims = behavior_dims.drop(index="HUMAN")
abs_diff_dims = (judge_dims - human_row).abs()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    abs_diff_dims,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "abs_diff vs. human"},
    ax=ax,
)
ax.set_title("|judge - human| per rubric dimension")
plt.tight_layout()
savefig(fig, "heatmap_dimension_abs_diff")
plt.show()

## Agreement by material category

`multi_llm_agreement_by_material_category.csv` — judge/human agreement broken down by target-compound-type / synthesis-method category. Using `abs_diff` here rather than `icc2`: per-category n is small (median 5, min 2 materials per judge x category cell), and at this n `icc2` is frequently undefined (NaN for ~1/3 of synthesis-method categories — insufficient variance for the underlying ANOVA). `abs_diff` is always computable but still has the scale-offset caveat discussed above, so treat this as a qualitative/exploratory breakdown; trust the pooled `loo_no_self` ICC2 numbers for the actual judge-selection argument.

We also show the raw mean scores (`l_mean` per judge, `h_mean` for human) with a `HUMAN` column appended, mirroring the extractor x judge panel above — this is the ground-truth reference score per category, not a 5th judge.

In [ ]:
by_category = load_csv("multi_llm_agreement_by_material_category.csv")

for category_type, group in by_category.groupby("category_type"):
    n_per_category = group.groupby("category")["n"].first()
    print(f"{category_type} -- n materials per category:")
    print(n_per_category.to_string())
    print()

    # abs_diff heatmap (judge vs. human distance)
    pivot = group.pivot(index="category", columns="judge", values="abs_diff")
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(pivot) + 2))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "abs_diff vs. human"},
        ax=ax,
    )
    ax.set_title(f"|judge - human| by {category_type}")
    plt.tight_layout()
    slug = category_type.lower().replace(" ", "_")
    savefig(fig, f"heatmap_agreement_by_{slug}")
    plt.show()

    # raw mean scores, with HUMAN reference column (h_mean is constant across
    # judges within a category, so any row's value works)
    means_pivot = group.pivot(
        index="category", columns="judge", values="l_mean"
    )
    means_pivot["HUMAN"] = group.groupby("category")["h_mean"].first()

    fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(means_pivot) + 2))
    sns.heatmap(
        means_pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "mean overall_score"},
        ax=ax,
    )
    ax.add_patch(
        plt.Rectangle(
            (len(means_pivot.columns) - 1, 0),
            1,
            len(means_pivot),
            fill=False,
            edgecolor=palette[2],
            lw=2.5,
        )
    )
    ax.set_title(f"Mean score by {category_type}, with human reference")
    plt.tight_layout()
    savefig(fig, f"heatmap_mean_score_by_{slug}_with_human")
    plt.show()

## Model choice: extractor x judge matrix with human reference (main-text candidate)

Same matrix as above, with a `HUMAN` column appended (`human_overall` from `insights_extractor_quality.csv`, indexed by extractor). This is not a 5th judge in the same sense as the 4 LLM columns — it's the one human judge's score of each extractor's output, so it only varies by row (extractor), not by column. It's appended as a column (not a row) because it's indexed on the same axis as the rows: "how did the human score this extractor," same question the 4 judge columns answer. Compare each row's `HUMAN` cell to its 4 LLM-judge cells to see which judges track human opinion most closely for that extractor, and compare the `HUMAN` column down all rows to confirm `claude-sonnet-4.6` is the top extractor by human judgment too.

In [ ]:
quality = load_csv("insights_extractor_quality.csv").set_index("extractor")

# human_overall is indexed by extractor (the human's score of that extractor's
# output), not by judge -- it belongs as an extra column, not a judge-row.
matrix_with_human = matrix.copy()
matrix_with_human["HUMAN"] = quality["human_overall"].reindex(matrix.index)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(
    matrix_with_human,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
# outline diagonal self-scoring cells
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
# outline the human reference column
ax.add_patch(
    plt.Rectangle(
        (len(matrix_with_human.columns) - 1, 0),
        1,
        len(matrix_with_human),
        fill=False,
        edgecolor=palette[2],
        lw=2.5,
    )
)
ax.set_title("Extractor performance across judges, with human reference")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge_with_human")
plt.show()

## Judge choice: bias-corrected agreement with human (SI)

`insights_judge_ranking_loo.csv`, `loo_no_self` rows only — agreement with human excluding each judge's self-scored cells, so self-preference bias can't inflate a judge's apparent quality (see Self-preference bias above). Ranked by `icc2`, which (unlike `abs_diff`) is robust to a judge's overall scale/offset.

In [ ]:
loo = load_csv("insights_judge_ranking_loo.csv")
loo_no_self = loo[loo["cell_set"] == "loo_no_self"].sort_values(
    "icc2", ascending=False
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=loo_no_self, x="judge", y="rho", ax=axes[0], color=palette[2])
axes[0].set_title("Rank correlation with human (rho)")
axes[0].set_ylabel("Spearman rho, loo_no_self")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=loo_no_self, x="judge", y="icc2", ax=axes[1], color=palette[0])
axes[1].set_title("Absolute agreement with human (ICC2)")
axes[1].set_ylabel("ICC2, loo_no_self")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig(fig, "bar_judge_agreement_loo_no_self")
plt.show()

## Human judge vs. LLM judges: per-dimension scoring behavior (SI)

`insights_judge_behavior.csv` includes a `HUMAN` row alongside the 4 LLM judges (each grading all 4 extractor LLMs) — this compares their scoring behavior directly, dimension by dimension, rather than only using the human as the reference for agreement metrics. Note the human judge's much larger `overall_std` (more discriminating / less clustered scores) than any LLM judge.

In [ ]:
behavior = load_csv("insights_judge_behavior.csv")
score_cols = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

heat = behavior.set_index("judge")[score_cols]
heat.columns = [c.replace("_score", "").replace("_", " ") for c in heat.columns]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    heat,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean score"},
    ax=ax,
)
for spine_row, judge in enumerate(heat.index):
    if judge == "HUMAN":
        ax.add_patch(
            plt.Rectangle(
                (0, spine_row),
                len(heat.columns),
                1,
                fill=False,
                edgecolor=palette[2],
                lw=2.5,
            )
        )
ax.set_title("Mean score per dimension: human vs. LLM judges")
plt.tight_layout()
savefig(fig, "heatmap_judge_behavior_dimensions")
plt.show()

## Extractor choice robustness: ranking agreement across graders (SI)

`insights_extractor_ranking_by_judge.csv` — Spearman correlation between each judge's (including HUMAN's) extractor ranking and the human ranking. High values across the board mean the extractor choice isn't an artifact of which judge you trust.

In [ ]:
ranking_by_judge = load_csv("insights_extractor_ranking_by_judge.csv")

fig, ax = plt.subplots(figsize=(7, 4))
order = ranking_by_judge.sort_values("spearman_vs_human", ascending=False)[
    "grader"
]
sns.barplot(
    data=ranking_by_judge,
    x="grader",
    y="spearman_vs_human",
    order=order,
    ax=ax,
    color=palette[2],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Spearman rho vs. human extractor ranking")
ax.set_title("Extractor ranking agreement with human, per grader")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig(fig, "bar_extractor_ranking_agreement")
plt.show()

## Concrete example: same extraction, four very different verdicts (main-text candidate, panel c)

Sourced directly from `annotations/cond-mat.0602418/` (not from the aggregate CSVs above) -- one extraction (`gemini-3-flash` extracting `Bi1.74Pb0.38Sr1.88CuO6+δ`), scored `overall_score` by all 5 graders. The extraction calls the synthesis method "flux growth"; the source says "floating-zone technique" -- a real, checkable error. It also invents a 2-step process and lists the target compound itself as a starting material for a paper that only mentions synthesis in passing (this is a characterization paper, not a synthesis paper).

- **HUMAN (4.5)**: docks half a point for the wrong method name and one structural nit; otherwise treats the extraction as good.
- **gemini-3-flash (4.9)**: doesn't catch the wrong method name at all -- praises the extraction for "not hallucinating" parameters. Systematically permissive judge.
- **deepseek-v3.2 (2.5)**: catches the same method error, but goes further and calls the *entire* structured process/materials list "hallucination" (even the correctly-null fields), effectively penalizing the extractor for using the schema at all. Systematically harsh judge.
- **claude-sonnet-4.6 / qwen3.5 (3.5 / 3.5)**: land between the two extremes.

This is why judge choice matters: gemini would rubber-stamp this real error, deepseek would over-penalize a mostly-correct extraction for structure it didn't even get wrong. Neither tracks the human's actual verdict as well as the "boring middle" judges do -- see the ICC2 ranking above.

In [ ]:
import json as _json
import textwrap as _textwrap
from pathlib import Path as _Path

EXAMPLE_PAPER = "cond-mat.0602418"
EXAMPLE_MATERIAL = "Bi1.74Pb0.38Sr1.88CuO6+\u03b4"
EXAMPLE_EXTRACTOR = "gemini-3-flash"

ann_dir = _Path("../../annotations") / EXAMPLE_PAPER
with open(ann_dir / "result.json") as fh:
    _result = _json.load(fh)
with open(ann_dir / "result_human.json") as fh:
    _human = _json.load(fh)

extractor_order = _human["extractor_order"]
extractor_idx = extractor_order.index(EXAMPLE_EXTRACTOR)

human_mat = next(
    m for m in _human["materials"] if m["material_name"] == EXAMPLE_MATERIAL
)
human_eval = human_mat["evaluations"][extractor_idx]["evaluation"]
human_recipe = human_mat["human_recipe"]

llm_entry = next(e for e in _result if e["synth_llm"] == EXAMPLE_EXTRACTOR)
mat_entry = next(
    m for m in llm_entry["materials"] if m["material"] == EXAMPLE_MATERIAL
)
extracted_synthesis = mat_entry["synthesis"]

verdicts = {"HUMAN": human_eval}
for jev in mat_entry["evaluations"]:
    verdicts[jev["judge_llm"]] = jev["evaluation"]

# --- Ground truth comparison: what's actually right/wrong, not just what the
# judges said about it. `human_recipe.steps[0].description` is the human
# annotator's verbatim quote of the source sentence -- the closest thing to
# "raw text" stored in these annotation files.
source_quote = human_recipe["steps"][0]["description"]
extracted_method = extracted_synthesis["synthesis_method"]
ground_truth_method = human_recipe["synthesis_method"]

print("SOURCE (human-quoted sentence from the paper):")
print(
    _textwrap.fill(
        source_quote, width=100, initial_indent="  ", subsequent_indent="  "
    )
)
print()
print(
    f"EXTRACTED synthesis_method ({EXAMPLE_EXTRACTOR}):  {extracted_method!r}"
)
print(
    f"GROUND TRUTH synthesis_method (human recipe):     {ground_truth_method!r}"
)
print(
    f"  -> extraction is WRONG: {extracted_method!r} != {ground_truth_method!r}"
)
print()

# --- Same wrong extraction, 5 verdicts on it
order = [
    "HUMAN",
    "gemini-3-flash",
    "claude-sonnet-4.6",
    "qwen3.5-397b-a17b",
    "deepseek-v3.2",
]
scores = [verdicts[j]["scores"]["overall_score"] for j in order]
colors = [palette[2] if j == "HUMAN" else palette[0] for j in order]

fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh(order, scores, color=colors)
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_xlim(0, 5.5)
ax.set_xlabel("overall_score")
ax.set_title(
    f'Same wrong extraction ("{extracted_method}" vs. true "{ground_truth_method}"): 5 verdicts'
)
ax.invert_yaxis()
plt.tight_layout()
savefig(fig, "bar_concrete_example_verdicts")
plt.show()

for j in order:
    print(f"--- {j} ({verdicts[j]['scores']['overall_score']}) ---")
    print(
        verdicts[j]["reasoning"]
        or verdicts[j]["scores"].get("overall_reasoning", "")
    )
    print()

## Candidate examples for panel (c) -- pick one

Same generic panel as above, applied to 5 more (paper, material, extractor) triples with a large gemini/deepseek spread, so you can compare and pick the strongest one for the main text. All pulled live from `annotations/`.

In [ ]:
import json as _json
import textwrap as _textwrap
from pathlib import Path as _Path

ANNOTATIONS_DIR = _Path("../../annotations")


def show_judge_disagreement_example(
    paper_id, material_name, extractor, save_name=None
):
    """Print source quote + extracted/ground-truth method + bar chart of all verdicts."""
    ann_dir = ANNOTATIONS_DIR / paper_id
    with open(ann_dir / "result.json") as fh:
        result = _json.load(fh)
    with open(ann_dir / "result_human.json") as fh:
        human = _json.load(fh)

    extractor_order = human["extractor_order"]
    extractor_idx = extractor_order.index(extractor)

    human_mat = next(
        m for m in human["materials"] if m["material_name"] == material_name
    )
    human_eval = human_mat["evaluations"][extractor_idx]["evaluation"]
    human_recipe = human_mat["human_recipe"]

    llm_entry = next(e for e in result if e["synth_llm"] == extractor)
    mat_entry = next(
        m for m in llm_entry["materials"] if m["material"] == material_name
    )
    extracted_synthesis = mat_entry["synthesis"]

    verdicts = {"HUMAN": human_eval}
    for jev in mat_entry["evaluations"]:
        verdicts[jev["judge_llm"]] = jev["evaluation"]

    source_quote = (
        human_recipe["steps"][0]["description"]
        if human_recipe["steps"]
        else "(no steps in human recipe)"
    )
    extracted_method = extracted_synthesis.get("synthesis_method")
    ground_truth_method = human_recipe.get("synthesis_method")

    print(f"=== {paper_id} | {material_name!r} extracted by {extractor} ===")
    print("SOURCE (human-quoted sentence from the paper):")
    print(
        _textwrap.fill(
            source_quote, width=100, initial_indent="  ", subsequent_indent="  "
        )
    )
    print()
    print(f"EXTRACTED synthesis_method ({extractor}):  {extracted_method!r}")
    print(
        f"GROUND TRUTH synthesis_method (human recipe):     {ground_truth_method!r}"
    )
    match = (
        "OK (matches)" if extracted_method == ground_truth_method else "WRONG"
    )
    print(f"  -> {match}")
    print()

    order = [
        j
        for j in [
            "HUMAN",
            "gemini-3-flash",
            "claude-sonnet-4.6",
            "qwen3.5-397b-a17b",
            "deepseek-v3.2",
        ]
        if j in verdicts
    ]
    scores = [verdicts[j]["scores"]["overall_score"] for j in order]
    colors = [palette[2] if j == "HUMAN" else palette[0] for j in order]

    fig, ax = plt.subplots(figsize=(7, 3.2))
    bars = ax.barh(order, scores, color=colors)
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_xlim(0, 5.5)
    ax.set_xlabel("overall_score")
    ax.set_title(f'"{material_name}" via {extractor} ({paper_id}): 5 verdicts')
    ax.invert_yaxis()
    plt.tight_layout()
    if save_name:
        savefig(fig, save_name)
    plt.show()

    for j in order:
        print(f"--- {j} ({verdicts[j]['scores']['overall_score']}) ---")
        print(
            verdicts[j]["reasoning"]
            or verdicts[j]["scores"].get("overall_reasoning", "")
        )
        print()

    return verdicts

In [ ]:
show_judge_disagreement_example(
    "64b40972b605c6803bd37ab4",
    "WFe2Ni-red",
    "deepseek-v3.2",
    save_name="bar_candidate_example_1",
)

In [ ]:
show_judge_disagreement_example(
    "1605.04038",
    "(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3",
    "gemini-3-flash",
    save_name="bar_candidate_example_2",
)

In [ ]:
show_judge_disagreement_example(
    "1706.00484",
    "SrTiO3",
    "gemini-3-flash",
    save_name="bar_candidate_example_3",
)

In [ ]:
show_judge_disagreement_example(
    "cond-mat.0503432",
    "GdCo2",
    "gemini-3-flash",
    save_name="bar_candidate_example_4",
)

In [ ]:
show_judge_disagreement_example(
    "1902.03049",
    "5-AGNR",
    "gemini-3-flash",
    save_name="bar_candidate_example_5",
)